# CA Exam Checker AI — Missing Material Recovery (CPU-safe)

This notebook repairs the **existing** archive from May 2024 through September 2026. It does not create a new archive and does not require a GPU.

**Sources:** CA Test Series archive pages are used as a secondary fallback for files whose BoS links were blocked. It downloads only direct PDF/Google Drive file links from MTP/RTP/PYQ/Suggested Answer tables. Folder links, unknown attempts, non-PDF pages and failed URLs are written to a review log rather than incorrectly saved as PDFs.

**Permission:** Run only for source categories that your written permission covers. The downloader reads the confirmations in the existing archive manifest; if missing, it will ask for confirmation. A link's being public is not, by itself, permission. Downloaded PDFs stay reference-only and are not automatically turned into training labels.


In [ ]:
import os, sys, subprocess, importlib.util, json, csv, re, time, hashlib, urllib.parse
from pathlib import Path
from datetime import datetime, timezone, date

if importlib.util.find_spec("bs4") is None or importlib.util.find_spec("fitz") is None or importlib.util.find_spec("gdown") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "beautifulsoup4==4.12.3", "PyMuPDF==1.26.4", "gdown==5.2.0"],
        check=True
    )

import fitz
import gdown
import requests
from bs4 import BeautifulSoup

from google.colab import drive
drive.mount("/content/drive")

ARCHIVE = Path(
    "/content/drive/MyDrive/CA_Exam_Checker_AI/training_runs/"
    "bos_reference_catalog/May2024_to_Sep2026_20261017T095236Z"
)
MANIFEST_PATH = ARCHIVE / "reference_manifest.json"
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Existing archive manifest not found: {MANIFEST_PATH}")

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
records = manifest.setdefault("files", [])
policy = manifest.get("source_policy", {})
START_DATE = date(2024, 5, 1)
END_DATE = date(2026, 9, 30)

# Do not silently assume permission if the original manifest did not record it.
official_ok = bool(policy.get("official_icai_bos_permission_confirmed_by_user", False))
secondary_ok = bool(policy.get("secondary_source_permission_confirmed_by_user", False))
if not official_ok:
    official_ok = input(
        "Written permission covers official ICAI/BoS material for this AI training/retrieval use? Type YES to confirm: "
    ).strip().upper() == "YES"
if not secondary_ok:
    secondary_ok = input(
        "Written permission/licensing covers the selected secondary-hosted copies for this AI training/retrieval use? Type YES to confirm: "
    ).strip().upper() == "YES"
if not official_ok and not secondary_ok:
    raise PermissionError("Neither official nor secondary source category was confirmed. No files downloaded.")

SOURCE_PAGES = [
    {
        "url": "https://www.catestseries.org/blogs/ca-foundation-icai-mock-test-papers-with-solution.php",
        "course": "CA_FOUNDATION",
        "fallback_type": "MTP",
    },
    {
        "url": "https://www.catestseries.org/blogs/ca-inter-mtp.php",
        "course": "CA_INTERMEDIATE",
        "fallback_type": "MTP",
    },
    {
        "url": "https://www.catestseries.org/blogs/icai-ca-inter-sample-papers-2022---download-pdfs-mtps-.php",
        "course": "CA_INTERMEDIATE",
        "fallback_type": "MTP",
    },
    {
        "url": "https://www.catestseries.org/blogs/download-ca-foundation-icai-ca-exams-rtps.php",
        "course": "CA_FOUNDATION",
        "fallback_type": "RTP",
    },
    {
        "url": "https://www.catestseries.org/blogs/ca-intermediate-rtp-revision-test-papers.php",
        "course": "CA_INTERMEDIATE",
        "fallback_type": "RTP",
    },
    {
        "url": "https://www.catestseries.org/blogs/download-ca-foundation-icai-ca-exams-suggested-answers.php",
        "course": "CA_FOUNDATION",
        "fallback_type": "SUGGESTED_ANSWER",
    },
    {
        "url": "https://www.catestseries.org/blogs/download-icai-ca-foundation-previous-year-question-papers.php",
        "course": "CA_FOUNDATION",
        "fallback_type": "PYQ",
    },
    {
        "url": "https://www.catestseries.org/blogs/download-icai-ca-inter-previous-year-question-papers.php",
        "course": "CA_INTERMEDIATE",
        "fallback_type": "PYQ",
    },
    {
        "url": "https://www.catestseries.org/blogs/rtps.php",
        "course": "CA_INTERMEDIATE",
        "fallback_type": "RTP",
    },
    {
        "url": "https://www.catestseries.org/blogs/ca-inter-past-exams.php",
        "course": "CA_INTERMEDIATE",
        "fallback_type": "PYQ",
    },
]

print("Archive:", ARCHIVE)
print("Existing manifest records:", len(records))
print("Official-source permission recorded/confirmed:", official_ok)
print("Secondary-source permission recorded/confirmed:", secondary_ok)
print("Source pages queued:", len(SOURCE_PAGES))


In [ ]:
# Discover direct file links from dated sections; skip unrelated links and folder links.
from collections import Counter, defaultdict

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; CAExamCheckerResearch/1.0; material-recovery)"
})

DIRECT_HOSTS = {
    "drive.google.com", "docs.google.com", "drive.usercontent.google.com",
    "resource.cdn.icai.org", "icai.org", "www.icai.org",
    "boslive.icai.org",
}
SECONDARY_HOSTS = {"catestseries.org", "www.catestseries.org"}
HEADINGS = {"h1", "h2", "h3", "h4", "h5", "h6"}

MONTHS = {
    "jan": 1, "january": 1, "feb": 2, "february": 2, "mar": 3, "march": 3,
    "apr": 4, "april": 4, "may": 5, "jun": 6, "june": 6, "jul": 7, "july": 7,
    "aug": 8, "august": 8, "sep": 9, "sept": 9, "september": 9,
    "oct": 10, "october": 10, "nov": 11, "november": 11, "dec": 12, "december": 12,
}
MONTH_LABELS = {
    1:"JANUARY",2:"FEBRUARY",3:"MARCH",4:"APRIL",5:"MAY",6:"JUNE",
    7:"JULY",8:"AUGUST",9:"SEPTEMBER",10:"OCTOBER",11:"NOVEMBER",12:"DECEMBER"
}
SUBJECT_MAP = {
    "CA_FOUNDATION": [
        (("accounting",), "ACCOUNTING"),
        (("business laws", "business law"), "BUSINESS_LAWS"),
        (("quantitative aptitude", "quantitative", "business mathematics"), "QUANTITATIVE_APTITUDE"),
        (("business economics", "economics"), "BUSINESS_ECONOMICS"),
    ],
    "CA_INTERMEDIATE": [
        (("advanced accounting", "adv acc"), "ADVANCED_ACCOUNTING"),
        (("corporate and other laws", "corporate laws", "corporate law"), "CORPORATE_AND_OTHER_LAWS"),
        (("taxation", "tax"), "TAXATION"),
        (("cost and management accounting", "costing", "cost accounting"), "COST_AND_MANAGEMENT_ACCOUNTING"),
        (("auditing and ethics", "auditing", "audit"), "AUDITING_AND_ETHICS"),
        (("financial management and strategic management", "fm sm", "financial management", "strategic management"), "FINANCIAL_MANAGEMENT_AND_STRATEGIC_MANAGEMENT"),
    ],
}

def host(url):
    return (urllib.parse.urlparse(url).hostname or "").lower()

def allowed_source_page(url):
    h = host(url)
    return urllib.parse.urlparse(url).scheme == "https" and any(h == x or h.endswith("." + x) for x in SECONDARY_HOSTS)

def allowed_direct_target(url):
    h = host(url)
    return urllib.parse.urlparse(url).scheme == "https" and any(h == x or h.endswith("." + x) for x in DIRECT_HOSTS)

def parse_attempt(text):
    x = re.sub(r"[_./-]+", " ", (text or "").lower())
    rx = r"\b(january|jan|february|feb|march|mar|april|apr|may|june|jun|july|jul|august|aug|september|sept|sep|october|oct|november|nov|december|dec)\.?\s*,?\s*(20\d{2})\b"
    matches = []
    for m in re.finditer(rx, x, re.I):
        month_num = MONTHS[m.group(1).lower()]
        y = int(m.group(2))
        matches.append((m.start(), date(y, month_num, 1)))
    rx2 = r"\b(20\d{2})\s+(january|jan|february|feb|march|mar|april|apr|may|june|jun|july|jul|august|aug|september|sept|sep|october|oct|november|nov|december|dec)\b"
    for m in re.finditer(rx2, x, re.I):
        month_num = MONTHS[m.group(2).lower()]
        y = int(m.group(1))
        matches.append((m.start(), date(y, month_num, 1)))
    if not matches:
        return "ATTEMPT_UNCONFIRMED", None
    _, d = min(matches, key=lambda z: z[0])
    return MONTH_LABELS[d.month] + "_" + str(d.year), d

def parse_series(text):
    x = (text or "").lower().replace("series –", "series ").replace("series -", "series ")
    m = re.search(r"\bseries\s*(?:no\.?\s*)?(i{1,3}|[123]|first|second|third)\b", x)
    if not m:
        return "SERIES_UNCONFIRMED"
    v = m.group(1)
    return {"i":"SERIES_I","ii":"SERIES_II","iii":"SERIES_III","1":"SERIES_I","2":"SERIES_II","3":"SERIES_III","first":"SERIES_I","second":"SERIES_II","third":"SERIES_III"}[v]

def infer_type(text, fallback):
    x = (text or "").lower()
    if "mtp" in x or "mock test" in x or "mock-test" in x:
        return "MTP"
    if "rtp" in x or "revision test" in x or "revision-test" in x:
        return "RTP"
    if "suggested answer" in x or "suggested answers" in x:
        return "SUGGESTED_ANSWER"
    if "previous year" in x or "past exam" in x or "pyq" in x:
        return "PYQ"
    return fallback

def infer_subject(text, course):
    x = re.sub(r"[^a-z0-9]+", " ", (text or "").lower()).strip()
    for tokens, label in SUBJECT_MAP[course]:
        if any(re.sub(r"[^a-z0-9]+", " ", token.lower()).strip() in x for token in tokens):
            return label
    # ICAI paper numbers sometimes precede the subject name.
    return "SUBJECT_UNKNOWN"

def direct_kind(url, link_text):
    h = host(url)
    path = urllib.parse.urlparse(url).path.lower()
    if h in {"drive.google.com", "docs.google.com", "drive.usercontent.google.com"}:
        if "/drive/folders/" in path:
            return "FOLDER"
        if "/file/d/" in path or "/uc" in path or "/document/d/" in path or re.search(r"[?&]id=", url):
            return "DRIVE_FILE"
    if ".pdf" in path and h in DIRECT_HOSTS:
        return "PDF"
    if h in {"icai.org", "www.icai.org", "boslive.icai.org"} and ".pdf" not in path:
        return "NAVIGATION_ONLY"
    return "OTHER"

def find_section_table(heading):
    node = heading.find_next()
    while node is not None:
        if node.name in HEADINGS:
            return None
        if node.name == "table":
            return node
        node = node.find_next()
    return None

def infer_role(header, cell_index, link_index):
    label = (header or "").lower()
    if "question" in label or "question paper" in label or label.strip() in {"qp", "paper"}:
        return "QUESTION"
    if "answer" in label or "solution" in label:
        return "ANSWER"
    if cell_index == 1:
        return "QUESTION"
    if cell_index >= 2:
        return "ANSWER"
    return "QUESTION" if link_index == 0 else "ANSWER"

candidates = []
folder_links = []
source_errors = []
visited_pages = 0

for source in SOURCE_PAGES:
    page_url = source["url"]
    if not allowed_source_page(page_url):
        source_errors.append({"url": page_url, "error": "Source page host not allowed."})
        continue
    try:
        response = SESSION.get(page_url, timeout=30, allow_redirects=True)
        if not allowed_source_page(response.url):
            raise ValueError("Source page redirected outside catestseries.org.")
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        visited_pages += 1
        page_title = soup.title.get_text(" ", strip=True) if soup.title else ""
        section_heads = soup.find_all(list(HEADINGS))
        for heading in section_heads:
            heading_text = " ".join(heading.get_text(" ", strip=True).split())
            heading_type = infer_type(heading_text, source["fallback_type"])
            # Only import recognized paper-type sections, never general marketing tables.
            if heading_type not in {"MTP", "RTP", "PYQ", "SUGGESTED_ANSWER"}:
                continue
            table = find_section_table(heading)
            if table is None:
                continue
            table_rows = table.find_all("tr")
            if len(table_rows) < 2:
                continue
            header_cells = table_rows[0].find_all(["th", "td"], recursive=False)
            header_labels = [re.sub(r"\s+", " ", c.get_text(" ", strip=True)).strip() for c in header_cells]
            section_context = heading_text + " " + page_title
            for row in table_rows[1:]:
                cells = row.find_all(["th", "td"], recursive=False)
                if not cells:
                    continue
                row_text = " ".join(c.get_text(" ", strip=True) for c in cells)
                if not row_text.strip():
                    continue
                course = source["course"]
                # Some combined pages mix levels; use an explicit course term when present.
                whole_context = (section_context + " " + row_text).lower()
                if "foundation" in whole_context:
                    course = "CA_FOUNDATION"
                elif "intermediate" in whole_context or "ca inter" in whole_context:
                    course = "CA_INTERMEDIATE"
                subject = infer_subject(row_text, course)
                attempt_label, attempt_date = parse_attempt(section_context + " " + row_text)
                if attempt_date is None or not (START_DATE <= attempt_date <= END_DATE):
                    continue
                series = parse_series(section_context + " " + row_text)
                # Header context determines the role. For tables without headers, link order is used.
                for cell_index, cell in enumerate(cells):
                    anchors = cell.find_all("a", href=True)
                    for link_index, anchor in enumerate(anchors):
                        href = urllib.parse.urljoin(response.url, anchor.get("href", "")).split("#", 1)[0]
                        if not allowed_direct_target(href):
                            continue
                        kind = direct_kind(href, anchor.get_text(" ", strip=True))
                        if kind == "NAVIGATION_ONLY" or kind == "OTHER":
                            continue
                        header = header_labels[cell_index] if cell_index < len(header_labels) else ""
                        role = infer_role(header, cell_index, link_index)
                        candidate = {
                            "course": course,
                            "attempt": attempt_label,
                            "series": series,
                            "date_status": "IN_RANGE" if attempt_date else "NEEDS_ATTEMPT_REVIEW",
                            "type": heading_type,
                            "subject": subject,
                            "role": role,
                            "url": href,
                            "source_tier": "OFFICIAL_ICAI_BOS" if host(href) in {"resource.cdn.icai.org", "icai.org", "www.icai.org", "boslive.icai.org"} else "SECONDARY_UNVERIFIED",
                            "discovered_on": response.url,
                            "section_heading": heading_text[:300],
                            "row_context": row_text[:500],
                            "direct_kind": kind,
                        }
                        if kind == "FOLDER":
                            folder_links.append(candidate)
                        else:
                            candidates.append(candidate)
        time.sleep(0.15)
    except Exception as exc:
        source_errors.append({"url": page_url, "error": f"{type(exc).__name__}: {exc}"})

# Remove duplicate link rows, keeping the more complete metadata if source pages repeat a link.
def drive_id(url):
    u = urllib.parse.urlparse(url)
    m = re.search(r"/file/d/([^/]+)", u.path)
    if m:
        return m.group(1)
    qs = urllib.parse.parse_qs(u.query)
    if qs.get("id"):
        return qs["id"][0]
    m = re.search(r"/document/d/([^/]+)", u.path)
    return m.group(1) if m else None

def url_key(url):
    gid = drive_id(url)
    return ("drive:" + gid) if gid else url.split("?", 1)[0]

deduped = {}
for candidate in candidates:
    key = url_key(candidate["url"])
    if key not in deduped:
        deduped[key] = candidate
    else:
        current = deduped[key]
        # Prefer a confidently identified attempt/series/subject and a known role.
        for field in ("course", "attempt", "series", "type", "subject", "role", "date_status", "source_tier"):
            if current.get(field) in {"UNKNOWN", "ATTEMPT_UNCONFIRMED", "SERIES_UNCONFIRMED", "SUBJECT_UNKNOWN"} and candidate.get(field) not in {"UNKNOWN", "ATTEMPT_UNCONFIRMED", "SERIES_UNCONFIRMED", "SUBJECT_UNKNOWN"}:
                current[field] = candidate[field]
candidates = list(deduped.values())

# Exclude unsupported/mislabelled subject rows and out-of-window or non-PDF destinations.
candidates = [
    c for c in candidates
    if c["course"] in {"CA_FOUNDATION", "CA_INTERMEDIATE"}
    and c["type"] in {"MTP", "RTP", "PYQ", "SUGGESTED_ANSWER"}
    and c["subject"] != "SUBJECT_UNKNOWN"
    and c["date_status"] == "IN_RANGE"
]
candidates.sort(key=lambda c: (c["course"], c["attempt"], c["type"], c["series"], c["subject"], c["role"]))

# Existing manifest/file links prevent repeated downloads.
existing_keys = set()
existing_shas = set()
for record in records:
    existing_url = record.get("url") or record.get("source_url") or ""
    if existing_url:
        existing_keys.add(url_key(existing_url))
    sha = record.get("sha256")
    if sha:
        existing_shas.add(sha)
    local_path = record.get("saved_pdf") or record.get("stored_pdf")
    if local_path and Path(local_path).is_file():
        try:
            h = hashlib.sha256(Path(local_path).read_bytes()).hexdigest()
            existing_shas.add(h)
        except Exception:
            pass

recovery_dir = ARCHIVE / "reference_pdfs" / "in_date_window" / "secondary_catestseries_recovery"
recovery_dir.mkdir(parents=True, exist_ok=True)
failed_rows = []
recovery_records = []
MAX_NEW_FILES = 250
MAX_FILE_BYTES = 80 * 1024 * 1024
MAX_TOTAL_BYTES = 8 * 1024 * 1024 * 1024
total_new_bytes = 0
new_count = 0
skip_count = 0

print(f"✅ Source pages read: {visited_pages}")
print(f"✅ Unique direct file candidates (May 2024–Sep 2026): {len(candidates)}")
print(f"📁 Folder links needing a separate folder-level review: {len(folder_links)}")
print("Starting targeted direct-file import (no whole-site re-crawl)...")

for index, item in enumerate(candidates, start=1):
    key = url_key(item["url"])
    if key in existing_keys:
        skip_count += 1
        continue
    # Confirm permission by the actual source host; a file on Drive is a secondary-hosted copy.
    if item["source_tier"] == "OFFICIAL_ICAI_BOS" and not official_ok:
        failed_rows.append({**item, "status": "SKIPPED_PERMISSION", "error": "Official-source permission not confirmed."})
        continue
    if item["source_tier"] == "SECONDARY_UNVERIFIED" and not secondary_ok:
        failed_rows.append({**item, "status": "SKIPPED_PERMISSION", "error": "Secondary-source permission not confirmed."})
        continue
    safe = lambda x: re.sub(r"[^A-Za-z0-9._-]+", "_", str(x)).strip("._-")[:80] or "UNKNOWN"
    filename = "__".join([
        safe(item["course"]), safe(item["attempt"]), safe(item["series"]),
        safe(item["subject"]), safe(item["type"]), safe(item["role"]),
        hashlib.sha256(item["url"].encode()).hexdigest()[:10]
    ]) + ".pdf"
    pdf_path = recovery_dir / filename
    try:
        if item["direct_kind"] == "DRIVE_FILE":
            result = gdown.download(url=item["url"], output=str(pdf_path), quiet=True, fuzzy=True)
            if not result or not pdf_path.exists():
                raise RuntimeError("gdown did not return a file.")
            raw = pdf_path.read_bytes()
        else:
            with SESSION.get(item["url"], timeout=45, stream=True, allow_redirects=True) as resp:
                if not allowed_direct_target(resp.url):
                    raise ValueError("PDF redirected outside the configured host allow-list.")
                resp.raise_for_status()
                if int(resp.headers.get("content-length") or 0) > MAX_FILE_BYTES:
                    raise ValueError("File exceeds 80 MB limit.")
                chunks, size = [], 0
                for chunk in resp.iter_content(1024 * 1024):
                    if not chunk:
                        continue
                    size += len(chunk)
                    if size > MAX_FILE_BYTES:
                        raise ValueError("File exceeds 80 MB limit.")
                    if total_new_bytes + size > MAX_TOTAL_BYTES:
                        raise ValueError("Recovery total-size limit reached.")
                    chunks.append(chunk)
                raw = b"".join(chunks)
            if not raw.startswith(b"%PDF"):
                raise ValueError(f"Target didn't return a PDF (content-type={resp.headers.get('content-type')}).")
            pdf_path.write_bytes(raw)
        if not raw.startswith(b"%PDF"):
            raise ValueError("Downloaded response is not a PDF; it will not be saved as a valid material.")
        sha = hashlib.sha256(raw).hexdigest()
        if sha in existing_shas:
            pdf_path.unlink(missing_ok=True)
            skip_count += 1
            existing_keys.add(key)
            continue
        doc = fitz.open(stream=raw, filetype="pdf")
        text = "\n\n".join(f"--- Page {p.number + 1} ---\n{p.get_text('text')}" for p in doc).strip()
        text_path = pdf_path.with_suffix(".txt")
        if len(text) >= 100:
            text_path.write_text(text, encoding="utf-8")
            extraction_status = "TEXT_EXTRACTED"
        else:
            extraction_status = "OCR_REQUIRED_OR_EMPTY_TEXT"
        record = {
            **item,
            "download_status": "DOWNLOADED",
            "saved_pdf": str(pdf_path),
            "sha256": sha,
            "bytes": len(raw),
            "page_count": len(doc),
            "extracted_text_path": str(text_path) if extraction_status == "TEXT_EXTRACTED" else None,
            "extracted_characters": len(text),
            "extraction_status": extraction_status,
            "training_status": "REFERENCE_ONLY_NOT_A_SUPERVISED_LABEL",
            "imported_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        records.append(record)
        recovery_records.append(record)
        existing_keys.add(key)
        existing_shas.add(sha)
        total_new_bytes += len(raw)
        new_count += 1
        print(f"{new_count:03d} ✅ {item['course']} | {item['attempt']} | {item['series']} | {item['subject']} | {item['type']} | {item['role']}")
    except Exception as exc:
        try:
            if pdf_path.exists() and not pdf_path.read_bytes().startswith(b"%PDF"):
                pdf_path.unlink()
        except Exception:
            pass
        failed_rows.append({**item, "status": "FAILED", "error": f"{type(exc).__name__}: {exc}"})
        print(f"⚠️ Failed: {item['course']} | {item['attempt']} | {item['subject']} | {exc}")
    time.sleep(0.1)

manifest["files"] = records
manifest["last_secondary_recovery"] = {
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "scope": "CA Foundation + CA Intermediate, May 2024 to September 2026",
    "source_pages": [s["url"] for s in SOURCE_PAGES],
    "new_downloads": new_count,
    "duplicate_skips": skip_count,
    "failed_or_skipped": len(failed_rows),
    "folder_links_for_review": len(folder_links),
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

recovery_csv = ARCHIVE / "secondary_recovery_downloads.csv"
fields = ["course","attempt","series","type","subject","role","source_tier","url","saved_pdf","extraction_status","download_status","error"]
with recovery_csv.open("w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    for r in recovery_records:
        writer.writerow({k:r.get(k,"") for k in fields})

failed_csv = ARCHIVE / "secondary_recovery_failures.csv"
with failed_csv.open("w", newline="", encoding="utf-8-sig") as f:
    fields2 = ["course","attempt","series","type","subject","role","source_tier","url","discovered_on","status","error"]
    writer = csv.DictWriter(f, fieldnames=fields2)
    writer.writeheader()
    for r in failed_rows:
        writer.writerow({k:r.get(k,"") for k in fields2})

folder_csv = ARCHIVE / "secondary_recovery_folder_links.csv"
with folder_csv.open("w", newline="", encoding="utf-8-sig") as f:
    fields3 = ["course","attempt","series","type","subject","role","url","discovered_on"]
    writer = csv.DictWriter(f, fieldnames=fields3)
    writer.writeheader()
    for r in folder_links:
        writer.writerow({k:r.get(k,"") for k in fields3})

print("\n✅ Targeted recovery finished.")
print("New PDFs:", new_count)
print("Duplicates skipped:", skip_count)
print("Failed/skipped links:", len(failed_rows))
print("Folder links needing separate review:", len(folder_links))
print("Recovery manifest:", MANIFEST_PATH)
print("Download log:", recovery_csv)
print("Failure log:", failed_csv)
print("Folder review:", folder_csv)


In [ ]:
# Recheck full archive coverage after the recovery cell.
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
downloaded = [
    r for r in manifest.get("files", [])
    if r.get("download_status") == "DOWNLOADED"
]
from collections import Counter, defaultdict

course_type = Counter((r.get("course","UNKNOWN"), r.get("type","OTHER")) for r in downloaded)
print("Total manifest downloads:", len(downloaded))
print("\nCourse + type:")
for (course, material_type), count in sorted(course_type.items()):
    print(f"{course:20} | {material_type:24} | {count}")

# Pair audits are grouped by course, attempt, series, subject and material type.
pairs = defaultdict(set)
for r in downloaded:
    if r.get("type") == "MTP":
        key = (
            r.get("course","UNKNOWN"), r.get("attempt","UNKNOWN"),
            r.get("series","UNKNOWN"), r.get("subject","UNKNOWN")
        )
        role = r.get("role")
        if role in {"QUESTION","ANSWER"}:
            pairs[key].add(role)

pair_status = Counter(
    "PAIR_COMPLETE" if roles == {"QUESTION","ANSWER"}
    else "QUESTION_ONLY" if "QUESTION" in roles
    else "ANSWER_ONLY" if "ANSWER" in roles
    else "NO_ROLE_METADATA"
    for roles in pairs.values()
)
print("\nMTP question/answer pair status:", pair_status)

pair_path = ARCHIVE / "mtp_question_answer_pairs_after_recovery.csv"
with pair_path.open("w", newline="", encoding="utf-8-sig") as f:
    fields = ["course","attempt","series","subject","status"]
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    for (course,attempt,series,subject), roles in sorted(pairs.items()):
        status = (
            "PAIR_COMPLETE" if roles == {"QUESTION","ANSWER"}
            else "QUESTION_ONLY" if "QUESTION" in roles
            else "ANSWER_ONLY" if "ANSWER" in roles
            else "NO_ROLE_METADATA"
        )
        writer.writerow({"course":course,"attempt":attempt,"series":series,"subject":subject,"status":status})

print("\nMTP pairing audit:", pair_path)
print("Note: links not found, folder-only links, and website 403/404 responses remain in the recovery failure/review files.")
